# MCNNM Non-Negative Baseline Projection

This notebook shows `baseline_projection="clip_nonnegative"` for `MCNNMPanelSolver`. The projection is a post-estimation support correction on the final baseline. Raw baseline and raw tau remain available.

In [1]:
from pathlib import Path
import sys

import numpy as np

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from causaltensor.cauest.MCNNM import MCNNMPanelSolver

Generate a small intermittent-outcome panel where the unconstrained fitted baseline can dip below zero.

In [2]:
rng = np.random.default_rng(1)
shape = (12, 12)
rank = 2

U = rng.normal(size=(shape[0], rank))
V = rng.normal(size=(shape[1], rank))
baseline_true = U @ V.T
baseline_true = baseline_true - np.min(baseline_true) + 0.05
baseline_true = np.maximum(baseline_true - np.quantile(baseline_true, 0.35), 0)

X = rng.normal(size=shape)
Z = np.zeros(shape)
Z[shape[0] // 2:, shape[1] // 2:] = 1

tau_true = 0.2
O = baseline_true + 0.1 * X + tau_true * Z + rng.normal(scale=0.2, size=shape)

Fit MCNNM and request projected companion outputs.

In [3]:
solver = MCNNMPanelSolver(Z=Z, X=X)
res = solver.fit(O=O, l=1.0, max_iter=500, baseline_projection="clip_nonnegative")

print("raw baseline min:", round(float(np.min(res.baseline)), 4))
print("projected baseline min:", round(float(np.min(res.baseline_projected)), 4))
print("raw tau:", round(float(res.tau), 4))
print("projected tau:", round(float(res.tau_projected), 4))
print("projection diagnostics:")
for key in ["clipped_fraction", "clipped_mass", "max_clipped_magnitude", "tau_shift"]:
    print(f"  {key}: {res.projection_diagnostics[key]:.4f}")

raw baseline min: -0.3203
projected baseline min: 0.0
raw tau: 0.3272
projected tau: 0.2888
projection diagnostics:
  clipped_fraction: 0.2569
  clipped_mass: 4.9934
  max_clipped_magnitude: 0.3203
  tau_shift: -0.0384
